In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("tugas11") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

26/11/19 11:41:03 WARN Utils: Your hostname, localhost resolves to a loopback address: 127.0.0.1; using 172.22.21.151 instead (on interface wlo1)
26/11/19 11:41:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 11:41:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/19 11:41:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
df_tugas = spark.read.csv("/home/plont/Documents/Praktikum_Big_Data/modul/tugas11_penjualan.csv", header=True, inferSchema=True)


A. Feature Engineering (bobot 20%)

Gunakan StringIndexer untuk mengubah kolom nama_menu, cabang, dan waktu_pesan menjadi indeks numerik. Gabungkan seluruh fitur (qty, harga_satuan, dan ketiga kolom hasil indexing) menjadi satu kolom features menggunakan VectorAssembler.

In [7]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

df_indexed = indexer_menu.fit(df_tugas).transform(df_tugas)
df_indexed = indexer_cabang.fit(df_indexed).transform(df_indexed)
df_indexed = indexer_waktu.fit(df_indexed).transform(df_indexed)

assembler = VectorAssembler(
    inputCols=["nama_menu_idx", "cabang_idx", "waktu_pesan_idx" , "qty", "harga_satuan"],
    outputCol="features" 
)

df_final = assembler.transform(df_indexed)
df_final.select("features").show(5, truncate=False)

+-------------------------+
|features                 |
+-------------------------+
|[5.0,2.0,1.0,1.0,30000.0]|
|[5.0,1.0,3.0,3.0,20000.0]|
|[5.0,2.0,2.0,5.0,22000.0]|
|[5.0,1.0,0.0,2.0,20000.0]|
|[4.0,0.0,1.0,2.0,30000.0]|
+-------------------------+
only showing top 5 rows



B. Train/Test Split & Pelatihan Model (bobot 20%)

Bagi data menjadi 75% training dan 25% testing (seed=101). Latih model LinearRegression dengan labelCol="total_penjualan".

In [9]:
from pyspark.ml.regression import LinearRegression

train, test = df_final.randomSplit([0.75, 0.25], seed=101)

lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")

model = lr.fit(train)

C. Evaluasi Model (bobot 20%)

Hitung dan tampilkan RMSE dan R² dari model pada data testing. Tulis 1-2 kalimat interpretasi: apakah model ini cukup baik untuk dipakai memprediksi omzet?

In [11]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = model.transform(test)
predictions.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction").show(10)

evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)

evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}  ({r2*100:.1f}% variasi data berhasil dijelaskan model)")

+---------+--------+-----------+---+------------+------------------+
|nama_menu|  cabang|waktu_pesan|qty|harga_satuan|        prediction|
+---------+--------+-----------+---+------------+------------------+
|Americano|Magelang|      Malam|  1|       25000|31056.146105506894|
|Americano|Magelang|      Malam|  1|       25000|31056.146105506894|
|Americano|Magelang|      Malam|  4|       22000| 88127.37331410217|
|Americano|Magelang|      Malam|  4|       22000| 88127.37331410217|
|Americano|Magelang|      Malam|  4|       30000|112310.74025938754|
|Americano|Magelang|      Malam|  5|       22000|110174.03658512792|
|Americano|Magelang|      Malam|  5|       25000|119242.79918960996|
|Americano|Magelang|       Pagi|  2|       30000| 67877.77492201283|
|Americano|Magelang|       Pagi|  3|       15000|44580.625170628526|
|Americano|Magelang|       Pagi|  4|       15000| 66627.28844165431|
+---------+--------+-----------+---+------------+------------------+
only showing top 10 rows

RMSE : R

D. Eksperimen Perbandingan Fitur (bobot 25%)

Latih model kedua yang HANYA menggunakan fitur qty dan harga_satuan (tanpa informasi menu/cabang/waktu). Bandingkan R² model kedua ini dengan model utama pada bagian C. Tulis kesimpulan: apakah menambahkan informasi menu/cabang/waktu benar-benar meningkatkan akurasi prediksi secara berarti?

In [26]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features" 
)

df_final = assembler.transform(df_tugas)

train, test = df_final.randomSplit([0.75, 0.25], seed=101)

lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")

model = lr.fit(train)

predictions = model.transform(test)
predictions.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction").show(10)

evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)

evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}  ({r2*100:.1f}% variasi data berhasil dijelaskan model)")

+---------+--------+-----------+---+------------+------------------+
|nama_menu|  cabang|waktu_pesan|qty|harga_satuan|        prediction|
+---------+--------+-----------+---+------------+------------------+
|Americano|Magelang|      Malam|  1|       25000| 31177.71885185805|
|Americano|Magelang|      Malam|  1|       25000| 31177.71885185805|
|Americano|Magelang|      Malam|  4|       22000| 88260.08378640366|
|Americano|Magelang|      Malam|  4|       22000| 88260.08378640366|
|Americano|Magelang|      Malam|  4|       30000|112428.48226630977|
|Americano|Magelang|      Malam|  5|       22000|110308.58857457379|
|Americano|Magelang|      Malam|  5|       25000|119371.73800453857|
|Americano|Magelang|       Pagi|  2|       30000| 68331.47268996951|
|Americano|Magelang|       Pagi|  3|       15000|45064.230328315665|
|Americano|Magelang|       Pagi|  4|       15000| 67112.73511648581|
+---------+--------+-----------+---+------------+------------------+
only showing top 10 rows

RMSE : R

E. Prediksi Skenario Baru (bobot 15%)

Buat sebuah DataFrame baru berisi 3 skenario pesanan (data dummy) (boleh bebas kombinasi menu/cabang/waktu/qty/harga, namun harus melalui tahapan StringIndexer dan VectorAssembler yang sama seperti data training — perhatikan: gunakan model StringIndexer yang sama yang sudah di-fit() sebelumnya, jangan membuat StringIndexer baru). Tampilkan hasil prediksi total_penjualan untuk ketiga skenario tersebut.

In [55]:
from pyspark.sql.functions import col

data_baru = [
    ("Kopi Susu", "Magelang", "Pagi", 3, 20000),
    ("Roti Bakar", "Yogyakarta", "Sore", 2, 25000),
    ("Americano", "Semarang", "Malam", 4, 18000)
]
kolom = ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"]
df_skenario = spark.createDataFrame(data_baru, kolom)

df_skenario = indexer_menu.fit(df_tugas).transform(df_skenario)
df_skenario = indexer_cabang.fit(df_tugas).transform(df_skenario)
df_skenario = indexer_waktu.fit(df_tugas).transform(df_skenario)

df_skenario_final = assembler.transform(df_skenario)

hasil_prediksi = model.transform(df_skenario_final)

df_baru = df_skenario.withColumn("total_penjualan", col("qty") * col("harga_satuan"))
df_baru.show()

+----------+----------+-----------+---+------------+-------------+----------+---------------+---------------+
| nama_menu|    cabang|waktu_pesan|qty|harga_satuan|nama_menu_idx|cabang_idx|waktu_pesan_idx|total_penjualan|
+----------+----------+-----------+---+------------+-------------+----------+---------------+---------------+
| Kopi Susu|  Magelang|       Pagi|  3|       20000|          1.0|       0.0|            0.0|          60000|
|Roti Bakar|Yogyakarta|       Sore|  2|       25000|          4.0|       1.0|            3.0|          50000|
| Americano|  Semarang|      Malam|  4|       18000|          0.0|       2.0|            1.0|          72000|
+----------+----------+-----------+---+------------+-------------+----------+---------------+---------------+

